# Complete architecture validation

This notebook checks the final model's structure, causality, sparse MoE dispatch, MTP alignment, cached decoding, checkpoint integrity, and numerical finiteness.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch
import torch.nn.functional as F

from src.model import DeepSeekConfig, DeepSeekMoE, DeepSeekWithMTP, FinalTinyDeepSeek
from src.training.checkpoint import load_checkpoint, save_checkpoint

torch.manual_seed(24)

In [ ]:
config = DeepSeekConfig(
    vocab_size=128, max_seq_len=32, dim=64,
    n_layers=3, n_dense_layers=1, n_heads=4,
    inter_dim=128, moe_inter_dim=32,
    q_lora_rank=32, kv_lora_rank=16,
    qk_nope_head_dim=16, qk_rope_head_dim=8, v_head_dim=16,
    n_routed_experts=4, n_shared_experts=1,
    n_activated_experts=2,
)
model = FinalTinyDeepSeek(config).eval()
tokens = torch.randint(0, config.vocab_size, (2, 8))
logits, hidden = model(tokens, return_hidden=True)
print('layers:', len(model.layers))
print('hidden:', tuple(hidden.shape))
print('logits:', tuple(logits.shape))
assert torch.isfinite(logits).all()

## Causality and sparse dispatch

Changing future tokens must not change earlier logits. The sparse MoE result is compared with a dense reference that evaluates every expert and masks unselected contributions.

In [ ]:
first = torch.tensor([[10, 20, 30, 40, 50, 60, 70, 80]])
second = torch.tensor([[10, 20, 30, 40, 50, 91, 92, 93]])
with torch.no_grad():
    first_logits = model(first)
    second_logits = model(second)
torch.testing.assert_close(first_logits[:, :5], second_logits[:, :5], atol=1e-5, rtol=1e-5)

moe = DeepSeekMoE(config).eval()
hidden_input = torch.randn(2, 5, config.dim)
sparse = moe(hidden_input)
flat = hidden_input.reshape(-1, config.dim)
indices, weights = moe.router(flat)
reference = moe.experts.forward_shared(flat)
for expert_id, expert in enumerate(moe.experts.routed_experts):
    output = expert(flat)
    reference = reference + output * (weights * (indices == expert_id)).sum(dim=-1, keepdim=True)
torch.testing.assert_close(sparse, reference.reshape_as(sparse), atol=1e-6, rtol=1e-5)
print('causality and sparse dispatch passed')

In [ ]:
with torch.no_grad():
    full = model(torch.randint(0, config.vocab_size, (1, 12)))
tokens = torch.randint(0, config.vocab_size, (1, 12))
with torch.no_grad():
    full = model(tokens)
    model.prefill(tokens[:, :-1])
    cached = model.decode_step(tokens[:, -1:])
torch.testing.assert_close(cached, full[:, -1:], atol=2e-4, rtol=2e-4)
print('cached final-position error:', (cached - full[:, -1:]).abs().max().item())

## MTP and checkpoint integrity

The validation suite checks alignment, future-token isolation, gradients, and preservation of router balancing buffers.

In [ ]:
mtp_model = DeepSeekWithMTP(config, mtp_weight=0.3).eval()
original = torch.tensor([[10, 20, 30, 40, 50, 60]])
changed = original.clone()
changed[:, 2] = 99
with torch.no_grad():
    original_result = mtp_model(original)
    changed_result = mtp_model(changed)
torch.testing.assert_close(original_result['mtp_logits'][:, 0], changed_result['mtp_logits'][:, 0], atol=1e-5, rtol=1e-5)
print('MTP shapes:', tuple(original_result['main_logits'].shape), tuple(original_result['mtp_logits'].shape))
print('MTP first position is isolated from its target')

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / 'validation.pt'
    save_checkpoint(path, model, step=24)
    restored, metadata = load_checkpoint(path)
torch.testing.assert_close(restored(tokens), model(tokens))
print('checkpoint step:', metadata['step'])
print('checkpoint round trip passed')